In [ ]:
# Install Tiny-SIMA and the optional VLM backend into this kernel.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Leave room for the PyTorch VLM before importing JAX.
os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.35"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,vlm,cuda12" if has_cuda else "notebook,vlm"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
# Import Torchvision before Transformers. Transformers caches optional
# dependency availability on first import.
torch = importlib.import_module("torch")
torchvision = importlib.import_module("torchvision")
transformers = importlib.import_module("transformers")
outlines = importlib.import_module("outlines")
transformers_utils = importlib.import_module("transformers.utils")

print(
    "PyTorch:", torch.__version__,
    "Torchvision:", torchvision.__version__,
    "Transformers:", transformers.__version__,
    "Outlines: installed",
)
if not transformers_utils.is_torchvision_available():
    raise RuntimeError(
        "Transformers cached Torchvision as unavailable. Restart this "
        "kernel, then run this installation cell before any other cell."
    )

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("CUDA detected:", has_cuda)
print("JAX memory fraction:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])

# Phase 4: hierarchical VLM planner

This notebook evaluates a frozen VLM at skill boundaries. The model supplies the Say term (objective usefulness); the Can term comes from horizon-bounded reachability and, optionally, the executor's own calibrated critic. The planner never sees the private scripted reference plan. Start it in a fresh kernel so the JAX memory fraction is applied before JAX initializes.

The evaluation has three layers:

1. Critic-as-affordance calibration turns the executor's value estimates into success probabilities by probing every skill at every chain boundary.
2. Controlled-prefix evaluation isolates planner decisions by letting the scripted executor advance the world after each prediction.
3. Closed-loop evaluation connects the planner to the learned stochastic executor and measures objective success.

Symbolic preconditions alone cannot supply a useful Can term: they test whether a block exists anywhere on the 64x64 map, which is true in essentially every state, so the planner ranked unreachable skills above reachable ones. The ablation ladder below separates that fix from the executor-competence one.

In [ ]:
import json
import time

import jax
import matplotlib.pyplot as plt

from tiny_sima.config import load_vlm_planner_config
from tiny_sima.envs.chains import make_skill_chain_environment
from tiny_sima.envs.craftax import render_state
from tiny_sima.evaluation import (
    WOOD_TO_STONE_CHAIN,
    collect_affordance_samples,
    evaluate_affordance_calibration,
    evaluate_planner_prefixes,
    evaluate_skill_chain,
    fit_affordance_calibration,
    load_goal_conditioned_checkpoint,
    make_learned_skill_executor,
    run_vlm_smoke_test,
)
from tiny_sima.goals import (
    DEFAULT_REACHABILITY,
    SKILL_IDS,
    skill_reachability_report,
)
from tiny_sima.planner import (
    IsotonicCalibration,
    PlannerContext,
    SayCanPlanner,
    SequentialUtilityScorer,
    TransformersVLMBackend,
    VLMUtilityScorer,
    build_planner_prompt,
    build_planner_system_prompt,
    constant_feasibility,
    constant_utility,
    make_critic_affordance_scorer,
    reachability_feasibility,
    symbolic_feasibility,
)

print("JAX devices:", jax.devices())
print("Reachability view radius:", DEFAULT_REACHABILITY.view_radius)
print("Unknown-target feasibility:", DEFAULT_REACHABILITY.unknown_feasibility)

In [ ]:
CHECKPOINT = repo_root / "runs" / "skill-ppo-local-1m" / "checkpoints" / "final"
VLM_CONFIG_PATH = repo_root / "configs" / "vlm_planner_qwen3_2b.yaml"
CALIBRATION_PATH = (
    repo_root / "runs" / "skill-ppo-local-1m" / "affordance_calibration.json"
)

QUICK_EVAL = True
RUN_VLM_SMOKE_TEST = True
RUN_AFFORDANCE_CALIBRATION = True
RUN_CLOSED_LOOP = True
NUM_PREFIX_EPISODES = 1 if QUICK_EVAL else 8
NUM_CLOSED_LOOP_EPISODES = 2 if QUICK_EVAL else 16
# Each calibration episode probes all six skills at every chain boundary, so
# this is the slowest knob here: episodes x boundaries x skills rollouts.
NUM_CALIBRATION_EPISODES = 2 if QUICK_EVAL else 12
CALIBRATION_SEED = 7
SKILL_HORIZON = 64
EVALUATION_SEED = 51

vlm_config = load_vlm_planner_config(VLM_CONFIG_PATH)
print("VLM:", vlm_config.model_id)
print("Checkpoint exists:", CHECKPOINT.exists())

## Inspect the planner input

The VLM receives a rendered frame plus a compact text record. This cell is cheap and should be inspected before loading model weights.

In [ ]:
preview_environment = make_skill_chain_environment()
preview_state = preview_environment.reset(jax.random.PRNGKey(EVALUATION_SEED)).state
preview_prompt = build_planner_prompt(
    preview_state,
    context=PlannerContext(
        objective=WOOD_TO_STONE_CHAIN.objective,
    ),
    candidate_skills=SKILL_IDS,
)
preview_pixels = render_state(
    preview_state,
    vlm_config.environment_name,
    block_pixel_size=vlm_config.block_pixel_size,
)
plt.figure(figsize=(8, 6))
plt.imshow(preview_pixels)
plt.axis("off")
plt.show()
print("SYSTEM:\n", build_planner_system_prompt())
print("\nUSER:\n", preview_prompt)

In [ ]:
# Cheap grounding check: compare what the precondition knows against what is
# actually within reach. The precondition is true whenever a block exists
# anywhere on the 64x64 map, which is why it ranked unreachable skills first.
preview_context = PlannerContext(objective=WOOD_TO_STONE_CHAIN.objective)
preview_steps = skill_reachability_report(preview_state, DEFAULT_REACHABILITY)
preview_graded = reachability_feasibility.score_all_feasibility(
    preview_state,
    preview_context,
    SKILL_IDS,
)
print(f"{'skill':22s} {'precondition':>12s} {'steps':>7s} {'graded':>8s}")
for skill_id in SKILL_IDS:
    steps = preview_steps[skill_id]
    print(
        f"{skill_id.value:22s} "
        f"{symbolic_feasibility(preview_state, preview_context, skill_id):12.2f} "
        f"{'-' if steps is None else steps:>7} "
        f"{preview_graded[skill_id]:8.2f}"
    )

## End-to-end VLM smoke test

This performs one real multimodal generation and verifies constrained decoding, complete-ranking validation, and symbolic grounding before the longer evaluations run.

In [ ]:
backend = TransformersVLMBackend(vlm_config)
vlm_utility = VLMUtilityScorer(backend=backend, config=vlm_config)

if RUN_VLM_SMOKE_TEST:
    smoke_result = run_vlm_smoke_test(
        vlm_config,
        scorer=vlm_utility,
        seed=EVALUATION_SEED,
    )
    print(smoke_result.to_dict())

## Critic as affordance model

The skill environment pays one sparse reward on success and terminates there, so the goal-conditioned critic already estimates `E[gamma ** k] * P(success)`: monotone in success probability, but squashed by discounting, which makes raw values unusable as a SayCan factor. This cell probes every skill at every boundary, records whether the executor finished it, and fits an isotonic map from critic value to success probability.

`unreachable_successes` is the number worth watching: it counts skills the reachability predicate called unreachable that the executor completed anyway by walking out of the observation window. Every one of those is an episode a hard gate would have thrown away, which is why reachability grades rather than vetoes.

The fit is saved to `CALIBRATION_PATH` so a later session can rebuild the scorer with `critic_affordance_scorer_from_checkpoint(CHECKPOINT, calibration_path=CALIBRATION_PATH)` without re-probing.

In [ ]:
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing executor checkpoint: {CHECKPOINT}")

# One load shared by the affordance scorer and the closed-loop executor, so the
# critic being calibrated belongs to the policy under test.
checkpoint = load_goal_conditioned_checkpoint(CHECKPOINT)
executor = make_learned_skill_executor(checkpoint, deterministic=False)
print("gamma:", checkpoint.config.gamma)
print("training skill_horizon:", checkpoint.config.skill_horizon)

if RUN_AFFORDANCE_CALIBRATION:
    uncalibrated_affordance = make_critic_affordance_scorer(checkpoint)
    started = time.perf_counter()
    affordance_samples = collect_affordance_samples(
        uncalibrated_affordance,
        executor,
        num_episodes=NUM_CALIBRATION_EPISODES,
        seed=CALIBRATION_SEED,
        skill_horizon=SKILL_HORIZON,
    )
    calibration = fit_affordance_calibration(affordance_samples)
    calibration.save(CALIBRATION_PATH)
    report = evaluate_affordance_calibration(affordance_samples, calibration)
    print(
        f"samples={report['num_samples']} "
        f"seconds={time.perf_counter() - started:.1f}"
    )
    print("overall:", json.dumps(report["overall"], indent=2, default=float))
    print(
        f"unreachable_rate={report['unreachable_rate']:.3f} "
        f"unreachable_successes={report['unreachable_successes']} "
        f"reachable_success_rate={report['reachable_success_rate']:.3f} "
        f"unreachable_success_rate={report['unreachable_success_rate']:.3f}"
    )
    print(f"\n{'skill':22s} {'mean_value':>10s} {'success':>8s} {'auc':>8s}")
    for skill_name, metrics in report["per_skill"].items():
        print(
            f"{skill_name:22s} {metrics['mean_critic_value']:10.3f} "
            f"{metrics['success_rate']:8.3f} {metrics['auc']:8.3f}"
        )
else:
    calibration = (
        IsotonicCalibration.load(CALIBRATION_PATH)
        if CALIBRATION_PATH.exists()
        else None
    )

calibrated_affordance = make_critic_affordance_scorer(
    checkpoint,
    calibration=calibration,
)

## Controlled-prefix planner ablation

The same randomized states are used for every planner. The VLM response cache means all VLM conditions share generations when their controlled states are identical, so the ladder below costs roughly one VLM pass per boundary.

The conditions isolate where grounding comes from:

| condition | Say | Can |
| --- | --- | --- |
| `scripted_oracle` | private reference sequence | reachability |
| `vlm_only` | VLM | none |
| `feasibility_only` | uniform | reachability |
| `vlm_plus_symbolic` | VLM | map-wide preconditions |
| `vlm_plus_reachability` | VLM | preconditions and reachability |
| `vlm_plus_critic_affordance` | VLM | reachability times calibrated critic |

Three metrics matter here beyond accuracy. `ungrounded_top` is the share of boundaries where a reachable skill existed and the VLM still ranked an unreachable one first - the failure being fixed, and it is identical across conditions that share a utility model because it measures the VLM alone. `rescued` is how often the Can term then recovered an accepted skill. `grounded` is how often the chosen skill was usable from that exact state, which the old precondition-based definition could not measure because it was satisfied almost everywhere.

In [ ]:
prefix_planners = {
    "scripted_oracle": SayCanPlanner(
        utility_scorer=SequentialUtilityScorer(WOOD_TO_STONE_CHAIN.reference_skills),
    ),
    "vlm_only": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=constant_feasibility,
    ),
    "feasibility_only": SayCanPlanner(utility_scorer=constant_utility),
    "vlm_plus_symbolic": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=symbolic_feasibility,
    ),
    "vlm_plus_reachability": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=reachability_feasibility,
    ),
    "vlm_plus_critic_affordance": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=calibrated_affordance,
    ),
}

prefix_results = {}
for name, planner in prefix_planners.items():
    started = time.perf_counter()
    result = evaluate_planner_prefixes(
        planner,
        num_episodes=NUM_PREFIX_EPISODES,
        seed=EVALUATION_SEED,
        skill_horizon=SKILL_HORIZON,
    )
    prefix_results[name] = result
    print(
        f"{name:27s} accepted={result['accepted_next_skill_accuracy']:.3f} "
        f"parse_valid={result['valid_utility_response_rate']:.3f} "
        f"valid={result['valid_selection_rate']:.3f} "
        f"grounded={result['grounded_selection_rate']:.3f} "
        f"ungrounded_top={result['ungrounded_top_choice_rate']:.3f} "
        f"rescued={result['feasibility_rescue_rate']:.3f} "
        f"reachable_skills={result['mean_feasible_skills']:.2f} "
        f"seconds={time.perf_counter() - started:.2f}"
    )

In [ ]:
INSPECT_CONDITION = "vlm_plus_critic_affordance"

for trace in prefix_results[INSPECT_CONDITION]["traces"]:
    selected = trace.selected_skill.value if trace.selected_skill else "none"
    accepted = ",".join(skill.value for skill in trace.accepted_skills)
    reachable = ",".join(skill.value for skill in trace.feasible_skills) or "none"
    ranking = " > ".join(skill.value for skill in trace.utility_ranking)
    print(
        f"episode={trace.episode} boundary={trace.boundary} "
        f"reference={trace.reference_skill.value:20s} selected={selected:20s} "
        f"accepted=[{accepted}] matched={trace.matched} "
        f"parse_valid={trace.utility_valid} "
        f"feasibility={trace.selected_feasibility:.2f}"
    )
    print(
        f"  reachable=[{reachable}] "
        f"top_feasible={trace.utility_top_feasible} "
        f"selected_feasible={trace.selected_feasible}"
    )
    if trace.utility_raw_response is not None:
        print(f"  ranking={ranking}")
        print(f"  raw={trace.utility_raw_response}")
    if trace.utility_error is not None:
        print(f"  error={trace.utility_error}")

## Closed-loop planner/executor ablation

This is intentionally small by default because VLM generation is sequential at option boundaries. Increase the episode count only after the controlled-prefix output is sensible. The executor is the one loaded for calibration above, run stochastically.

In [ ]:
if RUN_CLOSED_LOOP:
    closed_loop_results = {}
    for name, planner in prefix_planners.items():
        started = time.perf_counter()
        result = evaluate_skill_chain(
            executor,
            planner=planner,
            num_episodes=NUM_CLOSED_LOOP_EPISODES,
            seed=EVALUATION_SEED,
            skill_horizon=SKILL_HORIZON,
        )
        closed_loop_results[name] = result
        print(
            f"{name:27s} success={result['success_rate']:.3f} "
            f"skills={result['mean_completed_skills']:.2f} "
            f"steps={result['mean_environment_steps']:.1f} "
            f"seconds={time.perf_counter() - started:.2f} "
            f"terminations={result['termination_counts']}"
        )
        print(
            f"  final_failed_skills={result['final_failed_skill_counts']} "
            f"outcomes={result['final_failure_outcome_counts']}"
        )

## Interpretation

- Low accepted-skill accuracy with valid JSON is a planner reasoning or visual-grounding failure.
- Invalid utility responses are reported separately from states with no viable skill.
- `ungrounded_top` quantifies the VLM's grounding error on its own. It should be identical across the three `vlm_plus_*` conditions, because the Can term cannot change what the VLM ranked first; only `grounded` and `accepted` should move.
- `vlm_plus_symbolic` to `vlm_plus_reachability` measures how much of the fix comes from knowing what is in reach. `vlm_plus_reachability` to `vlm_plus_critic_affordance` measures how much comes from knowing what this executor can finish.
- Watch `termination_counts` for `no_viable_skill` in the closed loop. Reachability grades rather than vetoes precisely so that walking toward a distant resource stays available; a hard gate here turned four-episode scripted chains from 1.0 down to 0.5 success.
- Good prefix accuracy but poor closed-loop success points to executor recovery or planner adaptation after failures.
- If prompting is weak, preserve these zero-shot numbers before considering demonstrations or parameter-efficient fine-tuning.